In [1]:
import torch
import torch.nn.functional as F
from sympy.tensor import tensor
%pprint # to print arrays on one line


Pretty printing has been turned OFF


In [36]:
stories = open("C:\\dev\\tutorials\\LLM2\\gpt\\makemore\\data\\TinyStoriesV2-GPT4-valid.txt", encoding="utf-8").read()

In [41]:
context = 20
batch = 10

d = stories[:100]

ix = torch.randint(0, len(d) - context - 1, (batch,))
print(torch.tensor([d[start:start+context] for start in ix]))
print(ix)

TypeError: new(): invalid data type 'str'

In [ ]:
dic = sorted(set(ltr for str in stories for ltr in str))
dic.append("[EOT]")

stoi = {s: i + 1 for i, s in enumerate(dic)}
stoi['[EOT]'] = 0
itos = {i: s for s, i in stoi.items()}
dic

In [ ]:
len(dic)

In [15]:
B, T, C = 4, 8, 2

x = torch.tensor([i for i in  range(64)], dtype=torch.float).view(B, T, C)
print(x)

print(f"{x[0, 4:5]=}")
print(f"{torch.mean(x[0, 4:5], 0, dtype=torch.float)=}")

tensor([[[ 0.,  1.],
         [ 2.,  3.],
         [ 4.,  5.],
         [ 6.,  7.],
         [ 8.,  9.],
         [10., 11.],
         [12., 13.],
         [14., 15.]],

        [[16., 17.],
         [18., 19.],
         [20., 21.],
         [22., 23.],
         [24., 25.],
         [26., 27.],
         [28., 29.],
         [30., 31.]],

        [[32., 33.],
         [34., 35.],
         [36., 37.],
         [38., 39.],
         [40., 41.],
         [42., 43.],
         [44., 45.],
         [46., 47.]],

        [[48., 49.],
         [50., 51.],
         [52., 53.],
         [54., 55.],
         [56., 57.],
         [58., 59.],
         [60., 61.],
         [62., 63.]]])
x[0, 4:5]=tensor([[8., 9.]])
torch.mean(x[0, 4:5], 0, dtype=torch.float)=tensor([8., 9.])


In [16]:
xbow = torch.zeros((B,T,C))
for b in range(B):
  for t in range(T):
    xprev = x[b,: t+1]

In [17]:
a = torch.tensor([i for i in range(9)], dtype=torch.float).view(3, 3)
print(f"{a=}")
print(f"{torch.sum(a, 1, keepdim=True)}")
print(f"{a / torch.sum(a, 1, keepdim=True)}")

a=tensor([[0., 1., 2.],
        [3., 4., 5.],
        [6., 7., 8.]])
tensor([[ 3.],
        [12.],
        [21.]])
tensor([[0.0000, 0.3333, 0.6667],
        [0.2500, 0.3333, 0.4167],
        [0.2857, 0.3333, 0.3810]])


In [18]:
from gpt import GPTConfig

config = GPTConfig(vocab_size=5, context=2, n_embd=3, n_head=1, n_layer=1)
w_q = torch.ones([3, 3])
w_k = torch.ones([3, 3]) * 2
w_v = torch.ones([3, 3]) * 3

print(f"{w_k=}")

x = torch.ones([2, 3]) * 0.5
print(f"{x=}")

q: torch.Tensor = x @ w_q #[B, T, head_size]
k: torch.Tensor = x @ w_k #[B, T, head_size]
v: torch.Tensor = x @ w_v #[B, T, head_size]
tril = torch.ones([config.context, config.context]).tril()
wei = q @ k.transpose(0, 1) #/ config.head_size ** 0.5 # [B, T, T]
masked = wei.masked_fill(tril == 0, -torch.inf)
soft = F.softmax(masked, -1)
out =  soft @ v # [B, T, head_size]
print(f"{q=},\n{k=},\n{v=},\n{tril=},\n{wei=},\n{soft=},\n{out=}")

w_k=tensor([[2., 2., 2.],
        [2., 2., 2.],
        [2., 2., 2.]])
x=tensor([[0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000]])
q=tensor([[1.5000, 1.5000, 1.5000],
        [1.5000, 1.5000, 1.5000]]),
k=tensor([[3., 3., 3.],
        [3., 3., 3.]]),
v=tensor([[4.5000, 4.5000, 4.5000],
        [4.5000, 4.5000, 4.5000]]),
tril=tensor([[1., 0.],
        [1., 1.]]),
wei=tensor([[13.5000, 13.5000],
        [13.5000, 13.5000]]),
soft=tensor([[1.0000, 0.0000],
        [0.5000, 0.5000]]),
out=tensor([[4.5000, 4.5000, 4.5000],
        [4.5000, 4.5000, 4.5000]])


In [20]:
res = torch.cat([q, k, v], -1)
print(res)

tensor([[1.5000, 1.5000, 1.5000, 3.0000, 3.0000, 3.0000, 4.5000, 4.5000, 4.5000],
        [1.5000, 1.5000, 1.5000, 3.0000, 3.0000, 3.0000, 4.5000, 4.5000, 4.5000]])


In [27]:
ones = torch.ones(9, dtype=torch.int).view((3, 3))
print(ones)
print(torch.tril(torch.ones(5, 5))[:3, :3])

tensor([[1, 1, 1],
        [1, 1, 1],
        [1, 1, 1]], dtype=torch.int32)
tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])


In [32]:
dp = torch.nn.Dropout(p=0.2)
ones = torch.tensor([i for i in range(15)], dtype=torch.float).view((3, 5))
print(dp(ones))


tensor([[ 0.0000,  1.2500,  0.0000,  0.0000,  5.0000],
        [ 6.2500,  7.5000,  8.7500, 10.0000, 11.2500],
        [12.5000, 13.7500,  0.0000, 16.2500, 17.5000]])


In [13]:

emb1 = torch.tensor([i*10 for i in range(10)])
emb2 = torch.tensor([i*100 for i in range(10)])
emb = torch.stack((emb1, emb2)).T
print(emb)

x = torch.tensor([[0, 3, 5, 7], [1, 2, 3, 4]])
print(x.shape)
res = emb[x]
print(res)
res.shape

tensor([[  0,   0],
        [ 10, 100],
        [ 20, 200],
        [ 30, 300],
        [ 40, 400],
        [ 50, 500],
        [ 60, 600],
        [ 70, 700],
        [ 80, 800],
        [ 90, 900]])
torch.Size([2, 4])
tensor([[[  0,   0],
         [ 30, 300],
         [ 50, 500],
         [ 70, 700]],

        [[ 10, 100],
         [ 20, 200],
         [ 30, 300],
         [ 40, 400]]])


torch.Size([2, 4, 2])

In [35]:
batch = 2
context = 4
embs = 10
vocab_size = 10

x = torch.tensor([[0, 3, 5, 7], [4, 3, 2, 1]])
print(x.shape)
print(f"{x=}")

embeddings = torch.tensor([[i * (j+1) * 100 for i in range(embs)] for j in range(vocab_size)])
print(f"{embeddings=}")

p = torch.tensor([[i * (j+1) for i in range(embs)] for j in range(context)])
print(p.shape)
print(f"{p=}")

B,T = x.shape

idx = torch.tensor([i for i in range(T)])

print(f"p = {p[idx]=}")
print(f"embedded={embeddings[x]}")

print(f"res={embeddings[x] + p[idx]}")


torch.Size([2, 4])
x=tensor([[0, 3, 5, 7],
        [4, 3, 2, 1]])
embeddings=tensor([[   0,  100,  200,  300,  400,  500,  600,  700,  800,  900],
        [   0,  200,  400,  600,  800, 1000, 1200, 1400, 1600, 1800],
        [   0,  300,  600,  900, 1200, 1500, 1800, 2100, 2400, 2700],
        [   0,  400,  800, 1200, 1600, 2000, 2400, 2800, 3200, 3600],
        [   0,  500, 1000, 1500, 2000, 2500, 3000, 3500, 4000, 4500],
        [   0,  600, 1200, 1800, 2400, 3000, 3600, 4200, 4800, 5400],
        [   0,  700, 1400, 2100, 2800, 3500, 4200, 4900, 5600, 6300],
        [   0,  800, 1600, 2400, 3200, 4000, 4800, 5600, 6400, 7200],
        [   0,  900, 1800, 2700, 3600, 4500, 5400, 6300, 7200, 8100],
        [   0, 1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000]])
torch.Size([4, 10])
p=tensor([[ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9],
        [ 0,  2,  4,  6,  8, 10, 12, 14, 16, 18],
        [ 0,  3,  6,  9, 12, 15, 18, 21, 24, 27],
        [ 0,  4,  8, 12, 16, 20, 24, 28, 32, 36]])

In [3]:
f"{1.23456789:.3f}"

'1.235'